In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().parent
print(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

/Users/dhillo/Garage/trpo_repo_github_clone_for_code_context/trpo


In [2]:
import json
import math
import re

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from trpo_repro.config import load_config, save_config
from trpo_repro.rlhf.evaluate import run_before_after_eval
from trpo_repro.rlhf.metrics import read_jsonl

BASE_EVAL_CONFIG = Path('configs/rlhf/qwen25_05b_helpsteer3_eval.yaml')
FULL_EVAL_CONFIG = Path('configs/rlhf/qwen25_05b_helpsteer3_eval_full.yaml')
FULL_EVAL_DIR = Path('outputs/rlhf/qwen25_05b_helpsteer3_eval_full')

print('Base eval config:', BASE_EVAL_CONFIG.resolve())
print('Full eval config:', FULL_EVAL_CONFIG.resolve())
print('Full eval output dir:', FULL_EVAL_DIR.resolve())

Base eval config: /Users/dhillo/Garage/trpo_repo_github_clone_for_code_context/trpo/configs/rlhf/qwen25_05b_helpsteer3_eval.yaml
Full eval config: /Users/dhillo/Garage/trpo_repo_github_clone_for_code_context/trpo/configs/rlhf/qwen25_05b_helpsteer3_eval_full.yaml
Full eval output dir: /Users/dhillo/Garage/trpo_repo_github_clone_for_code_context/trpo/outputs/rlhf/qwen25_05b_helpsteer3_eval_full


/Users/dhillo/anaconda3/envs/garage/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
out_dir = FULL_EVAL_DIR
print(out_dir)

outputs/rlhf/qwen25_05b_helpsteer3_eval_full


In [4]:
FULL_JSONL = FULL_EVAL_DIR / 'before_after_samples.jsonl'
FULL_CSV = FULL_EVAL_DIR / 'before_after_samples.csv'
FULL_SUMMARY = FULL_EVAL_DIR / 'eval_summary.json'

rows = read_jsonl(FULL_JSONL)
df = pd.DataFrame(rows)

print('Loaded rows:', len(df))
print('Columns:', list(df.columns))

if FULL_SUMMARY.exists():
    summary = json.loads(FULL_SUMMARY.read_text(encoding='utf-8'))
    display(summary)
else:
    summary = {}

df.head(3)

Loaded rows: 2017
Columns: ['idx', 'domain', 'language', 'prompt', 'base_response', 'ppo_response', 'base_reward', 'ppo_reward', 'reward_delta', 'winner']


{'num_examples': 2017,
 'winner_counts': {'ppo': 1010, 'base': 1007},
 'domain_winner_counts': {'code': {'ppo': 203, 'base': 235},
  'general': {'ppo': 517, 'base': 414},
  'stem': {'base': 133, 'ppo': 112},
  'multilingual': {'base': 225, 'ppo': 178}},
 'base_reward': {'mean': -1.900489620842228,
  'median': -1.3385976552963257,
  'min': -23.949613571166992,
  'max': 23.105857849121094},
 'ppo_reward': {'mean': -1.3507756967006097,
  'median': -1.3307288885116577,
  'min': -18.38585662841797,
  'max': 21.954784393310547},
 'reward_delta': {'mean': 0.5497139241416182,
  'median': 0.008039474487304688,
  'min': -27.47940158843994,
  'max': 30.906216621398926},
 'base_response_chars': {'mean': 435.54040654437284,
  'median': 495.0,
  'min': 24.0,
  'max': 837.0},
 'ppo_response_chars': {'mean': 458.378284581061,
  'median': 508.0,
  'min': 26.0,
  'max': 833.0},
 'ppo_win_rate': 0.5007436787307883}

,idx,domain,language,prompt,base_response,ppo_response,base_reward,ppo_reward,reward_delta,winner
0,0,code,javascript_html_css,"<|im_start|>system\nYou are Qwen, created by A...","To convert a JSON object to a React component,...","Sure, I'd be happy to help you with that! Here...",0.577529,2.252626,1.675096,ppo
1,1,general,english,"<|im_start|>system\nYou are Qwen, created by A...","I'm sorry, but I can't assist with that. If yo...","I'm sorry, but I can't assist with that. If yo...",-11.093551,-6.698321,4.395230,ppo
2,2,code,sql,"<|im_start|>system\nYou are Qwen, created by A...",Sure! Let's start with a basic understanding o...,"Sure, I'd be happy to help you understand Post...",2.376246,-0.284226,-2.660472,base


In [5]:
def add_analysis_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ['prompt', 'base_response', 'ppo_response']:
        out[col] = out[col].fillna('').astype(str)
        out[f'{col}_chars'] = out[col].map(len)
        out[f'{col}_words'] = out[col].map(lambda x: len(x.split()))
    for col in ['base_reward', 'ppo_reward', 'reward_delta']:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out['abs_reward_delta'] = out['reward_delta'].abs()
    out['is_ppo_win'] = out['winner'].eq('ppo')
    out['is_base_win'] = out['winner'].eq('base')
    out['is_near_tie'] = out['reward_delta'].abs() <= 0.25
    return out

adf = add_analysis_columns(df)

# CJK / non-ASCII / simple pattern checks. These are heuristics, not final safety labels.
CJK_RE = re.compile(r'[㐀-䶿一-鿿豈-﫿]')
BAD_PATTERNS = [
    r'erot', r'adult', r' sex ', r'porn', r'cunt', r'busty', r'voyeur', r'hooker', r'libertin',
    r'blackColor', r'didReceiveMemoryWarning', r'SimpleName', r'HTTPHeader', r'numel',
]

def has_cjk(text):
    return bool(CJK_RE.search(str(text)))

def bad_hits(text):
    lower = ' ' + str(text).lower() + ' '
    return [pat for pat in BAD_PATTERNS if re.search(pat, lower)]

adf['ppo_has_cjk'] = adf['ppo_response'].map(has_cjk)
adf['base_has_cjk'] = adf['base_response'].map(has_cjk)
adf['ppo_bad_hits'] = adf['ppo_response'].map(bad_hits)
adf['base_bad_hits'] = adf['base_response'].map(bad_hits)
adf['ppo_bad_hit_count'] = adf['ppo_bad_hits'].map(len)
adf['base_bad_hit_count'] = adf['base_bad_hits'].map(len)

print('PPO CJK response count:', int(adf['ppo_has_cjk'].sum()))
print('PPO bad-pattern count:', int((adf['ppo_bad_hit_count'] > 0).sum()))

PPO CJK response count: 153
PPO bad-pattern count: 2


In [6]:
import html

EXAMPLE_CSS = """
<style>
.eval-example {
    color: #172033;
    background: #f3f6fb;
    border: 1px solid #ccd6e3;
    border-radius: 8px;
    padding: 18px;
    font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
}
.eval-header {
    display: flex;
    justify-content: space-between;
    gap: 16px;
    align-items: flex-start;
    border-bottom: 1px solid #ccd6e3;
    padding-bottom: 14px;
    margin-bottom: 14px;
}
.eval-kicker {
    color: #52627a;
    font-size: 12px;
    font-weight: 700;
    letter-spacing: .04em;
    text-transform: uppercase;
}
.eval-example h2 {
    margin: 3px 0 0;
    color: #111827;
    font-size: 22px;
}
.eval-chip-row {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    justify-content: flex-end;
}
.eval-meta {
    justify-content: flex-start;
    margin-bottom: 12px;
}
.eval-chip {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    color: #172033;
    background: #ffffff;
    border: 1px solid #c9d3df;
    border-radius: 999px;
    padding: 5px 10px;
    font-size: 13px;
    line-height: 1.35;
}
.eval-chip b {
    color: #52627a;
    font-weight: 700;
}
.eval-chip--base {
    background: #e8f1ff;
    border-color: #8fb5e8;
}
.eval-chip--ppo {
    background: #e8f8ef;
    border-color: #87c99d;
}
.eval-chip--delta {
    background: #fff4d8;
    border-color: #e7bd64;
}
.eval-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(320px, 1fr));
    gap: 12px;
}
.eval-card {
    border: 1px solid #c9d3df;
    border-left-width: 5px;
    border-radius: 8px;
    background: #ffffff;
    overflow: hidden;
    margin: 12px 0;
}
.eval-card--prompt {
    border-left-color: #d59c2f;
}
.eval-card--base {
    border-left-color: #3d7fd1;
}
.eval-card--ppo {
    border-left-color: #2e9d59;
}
.eval-card__title {
    color: #111827;
    background: #e9eef6;
    border-bottom: 1px solid #c9d3df;
    padding: 9px 12px;
    font-size: 13px;
    font-weight: 800;
}
.eval-card--prompt .eval-card__title {
    background: #fff0c7;
}
.eval-card--base .eval-card__title {
    background: #dceaff;
}
.eval-card--ppo .eval-card__title {
    background: #daf3e3;
}
.eval-card pre {
    margin: 0;
    padding: 13px 14px;
    color: #111827;
    background: #ffffff;
    white-space: pre-wrap;
    overflow-wrap: anywhere;
    font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
    font-size: 13px;
    line-height: 1.55;
    max-height: 560px;
    overflow-y: auto;
}
@media (max-width: 720px) {
    .eval-header {
        display: block;
    }
    .eval-chip-row {
        justify-content: flex-start;
        margin-top: 10px;
    }
}
</style>
"""

def _chip(label: str, value: object, tone: str = 'neutral') -> str:
    safe_label = html.escape(str(label))
    safe_value = html.escape(str(value))
    return f'<span class="eval-chip eval-chip--{tone}"><b>{safe_label}</b>{safe_value}</span>'


def _format_delta(value: object) -> str:
    value = float(value)
    sign = '+' if value >= 0 else ''
    return f'{sign}{value:.4f}'


def _box(title: str, text: str, tone: str = 'neutral') -> str:
    safe_title = html.escape(str(title))
    safe_text = html.escape(str(text or ''))
    return f'''
    <section class="eval-card eval-card--{tone}">
        <div class="eval-card__title">{safe_title}</div>
        <pre>{safe_text}</pre>
    </section>
    '''


def show_example(idx: int, source_df: pd.DataFrame = None):
    """Show complete prompt/base/PPO outputs for a row index from the eval file."""
    source_df = adf if source_df is None else source_df
    match = source_df[source_df['idx'].astype(int) == int(idx)]
    if match.empty:
        raise ValueError(f'No row with idx={idx}. Available idx range: {source_df.idx.min()}..{source_df.idx.max()}')
    row = match.iloc[0]
    winner = str(row.get('winner', 'unknown')).lower()
    winner_tone = 'ppo' if winner == 'ppo' else 'base' if winner == 'base' else 'neutral'
    base_reward = f"{float(row['base_reward']):.4f}"
    ppo_reward = f"{float(row['ppo_reward']):.4f}"
    header = f'''
    <div class="eval-example">
        <div class="eval-header">
            <div>
                <div class="eval-kicker">Full eval sample</div>
                <h2>Example {int(row['idx'])}</h2>
            </div>
            <div class="eval-chip-row">
                {_chip('Winner', row.get('winner', 'unknown'), winner_tone)}
                {_chip('Reward delta', _format_delta(row['reward_delta']), 'delta')}
            </div>
        </div>
        <div class="eval-chip-row eval-meta">
            {_chip('Domain', row.get('domain', 'unknown'))}
            {_chip('Language', row.get('language', 'unknown'))}
            {_chip('Base reward', base_reward, 'base')}
            {_chip('PPO reward', ppo_reward, 'ppo')}
            {_chip('Base chars', int(row['base_response_chars']))}
            {_chip('PPO chars', int(row['ppo_response_chars']))}
        </div>
    '''
    html_out = (
        EXAMPLE_CSS
        + header
        + _box('Prompt', row['prompt'], 'prompt')
        + '<div class="eval-grid">'
        + _box('Base Qwen response', row['base_response'], 'base')
        + _box('PPO-RLHF response', row['ppo_response'], 'ppo')
        + '</div></div>'
    )
    display(HTML(html_out))

# Try one example after full eval is loaded.
show_example(int(adf.iloc[0]['idx']))

In [10]:
show_example(int(adf.iloc[1300]['idx']))

In [12]:
show_example(int(adf.iloc[774]['idx']))

In [13]:
show_example(int(adf.iloc[1362]['idx']))

In [14]:
show_example(int(adf.iloc[205]['idx']))

In [15]:
show_example(int(adf.iloc[1663]['idx']))

In [ ]:
show_example(int(adf.iloc[0]['idx']))

In [ ]:
"""
1206 seems a good example...i have become a real estate agent hah
1300 is also a lovely example
"""